# Meeting 11, Act 2: The Same Layer in PyTorch, and on a GPU

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 11.

PyTorch is the library in which most neural networks are written today. Its arrays are called tensors, and it computes with them in nearly the notation of NumPy. It can also run the same computation on a second processor of the computer, the GPU. In this act your group writes the layer of Act 1 in PyTorch, moves it to a GPU, and measures the two processors against each other.

## How this lab works

- **This is the notebook of Act 2 of meeting 11.** It starts from its own setup cell and uses nothing that was computed in the notebook of Act 1, except the number of your group.
- **Work in the same group.** Each member opens this notebook and runs it. First open the **File** menu and choose **Save a copy in Drive**, so that your changes are kept.
- **Do Step 9 before anything else.** It asks Colab for a GPU, and Colab erases the work of a notebook when it changes its machine.
- **Run the cells in order**, from the top. To run a cell, click it and press **Shift + Enter**.
- A paragraph that begins with **Your turn** asks you to do something before you go on.
- A code cell under the word **Check** tests your work and prints a message. If the message reports a problem, fix it before you go on.
- In Step 11 you ask an **AI assistant** to translate a function. Paste the prompt you used into the cell marked **Our prompt**, so that the notebook records how the code was made.
- **In Step 15** your group posts its measurements on the discussion board of this meeting.
- **Keep this notebook open when Act 2 ends.** Act 3 asks for one of its numbers, and at the end of the meeting each member uploads the notebooks to Canvas.
- **After the meeting**, **Review Quiz 11** on Canvas asks about this lab. Where the notebook says that the quiz asks about a point, read that part again before you take the quiz.

**Your turn.** Double-click this cell and replace this line with the names of the members of your group.

## <font color="#1967d2"><b>Step 9: First, ask Colab for a GPU</b></font>

Later in this act the notebook uses a **GPU**, a second processor, which Step 13 describes. Colab gives a notebook a machine with a GPU only when the notebook asks for one. When Colab changes the machine of a notebook, it erases everything that the notebook has computed. So ask now, before anything is computed.

**Your turn.**

1. Run the next cell. On a machine without a GPU, its last line ends with `False`.
2. Open the **Runtime** menu and choose **Change runtime type**. Under **Hardware accelerator**, select **T4 GPU**, and choose **Save**.
3. Run the cell again. It now says `True`, and it names the GPU.

Colab lends GPUs for free only while it has some to spare. If it answers that no GPU is available, go on without one: every cell of this notebook also runs on a machine without a GPU, and one GPU in a group is enough. The cells that need a GPU say so.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("Does PyTorch see a GPU?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("The GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Setup. Run this cell after Step 9.
import ast
import inspect
import os
import textwrap
import time
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

# The address of the course's files for this unit, and of the folder that holds the images.
SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
IMAGES = SITE + "images/"

GPU = torch.cuda.is_available()          # True when this session has a GPU


def read_image(file):
    """Read an image file into a NumPy array with values from 0 to 1."""
    return np.array(Image.open(urlopen(IMAGES + file))) / 255.0


def show(pictures, titles):
    """Draw arrays as pictures: an array with two dimensions in grey, one with three in colour."""
    columns = 2 if len(pictures) == 4 else len(pictures)
    lines = len(pictures) // columns
    fig, axes = plt.subplots(lines, columns, figsize=(5 * columns, 4.2 * lines), squeeze=False)
    for ax, picture, title in zip(axes.flat, pictures, titles):
        ax.imshow(np.clip(picture, 0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


def reference_colours(X, K=5, rounds=10):
    """K colours that occur often among the rows of X, from the darkest to the lightest: the function of Act 1."""
    grey_weights = np.array([0.299, 0.587, 0.114])
    sample = X[::7]
    order = np.argsort(sample @ grey_weights)
    C = np.array([sample[part].mean(axis=0) for part in np.array_split(order, K)])
    for _ in range(rounds):
        distance = ((sample[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
        nearest = distance.argmin(axis=1)
        C = np.array([sample[nearest == j].mean(axis=0) if (nearest == j).any() else C[j] for j in range(K)])
    return C[np.argsort(C @ grey_weights)]


def has_loop(function):
    """True when the text of the function contains a loop."""
    try:
        tree = ast.parse(textwrap.dedent(inspect.getsource(function)))
    except (OSError, TypeError, SyntaxError):
        return False
    loops = (ast.For, ast.While, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    return any(isinstance(node, loops) for node in ast.walk(tree))


def mentions(function, words):
    """True when the text of the function contains one of the words."""
    try:
        text = inspect.getsource(function)
    except (OSError, TypeError):
        return False
    return any(word in text for word in words)


print("Setup is done. This session", "has a GPU." if GPU else "has no GPU.")

## <font color="#1967d2"><b>Step 10: From arrays to tensors</b></font>

**Your turn.** Every member types the group's number in the next cell, in place of the `0`, and runs the cell. The cell computes again, with NumPy, what Act 1 computed: the points `X`, one row for each pixel of your image; the weights `W` and the biases `b` of the layer of five units; and the scores `S`.

In [ ]:
IMAGE = 0        # replace 0 with your group's number, from 1 to 30

assert 1 <= IMAGE <= 30, "Type your group's number in place of the 0."

images = pd.read_csv(IMAGES + "images.csv", index_col="number")
picture_info = images.loc[IMAGE]
image = read_image(picture_info["file"])
height, width = image.shape[0], image.shape[1]

X = image.reshape(-1, 3)                 # the points: one row for each pixel
n, m = X.shape
K = 5
C = reference_colours(X, K)              # the reference colours
W = C.copy()                             # the weights of the layer: one row for each unit
b = -0.5 * (C ** 2).sum(axis=1)          # the biases of the layer: one number for each unit
S = X @ W.T + b                          # the scores: the line of Act 1

print(f"Image {IMAGE}: {picture_info['title']} ({picture_info['creator']})")
print("NumPy arrays.  X:", X.shape, "  W:", W.shape, "  b:", b.shape, "  S:", S.shape)

**The words.** Arrays are named after their number of dimensions.

| Name | Dimensions | Example in this lab | Shape |
|---|---|---|---|
| scalar | 0 | the bias of one unit | a single number |
| vector | 1 | one pixel; the weights of one unit | (3,) |
| matrix | 2 | the points `X`; the weights `W` of a layer | (n, 3) and (K, 3) |
| volume | 3 | a colour image | (height, width, 3) |
| (no special name) | 4 | a batch of colour images | (30, 96, 96, 3) |

All of them are **tensors**: a vector is a tensor with one dimension, a matrix is a tensor with two dimensions, and a colour image is a tensor with three. In mathematics, tensors with more than two dimensions have a theory of their own. In this course a tensor is only a data structure: an array of numbers with some number of dimensions.

PyTorch calls its arrays tensors. The next cell makes three of them. `torch.from_numpy` makes a tensor from a NumPy array, and `torch.tensor` makes one from a list.

In [ ]:
w_t = torch.tensor([0.299, 0.587, 0.114])        # the weights of the grey unit, from a list
X_t = torch.from_numpy(X)                        # the points, from the NumPy array X
image_t = torch.from_numpy(image)                # the image, from the NumPy array image

for name, tensor in [("w_t", w_t), ("X_t", X_t), ("image_t", image_t)]:
    print(f"{name:8s} dimensions: {tensor.ndim}   shape: {tuple(tensor.shape)}   type of its numbers: {tensor.dtype}")

**A first difference: the type of the numbers.** NumPy stores a decimal number in 8 bytes, a type called `float64`. PyTorch's own choice is `float32`: 4 bytes, with about 7 significant digits in place of about 16. That is enough for neural networks, it takes half the memory, and a GPU computes fastest with it.

`torch.from_numpy(X)` kept the type of the NumPy array, `float64`. The tensor `w_t` was made by PyTorch, with `float32`. PyTorch does not multiply tensors of two different types: the next cell shows the error, and then converts `X_t`.

In [ ]:
try:
    X_t @ w_t
except RuntimeError as error:
    print("X_t @ w_t fails:", error)

X_t = X_t.float()                                # the same numbers, as float32
megabytes = X_t.element_size() * X_t.nelement() / 1e6
print("after X_t = X_t.float(), the type of its numbers is", X_t.dtype)
print(f"memory of the points: {X.nbytes / 1e6:.1f} MB as float64, {megabytes:.1f} MB as float32")

s_t = X_t @ w_t                                  # one unit, many points: the line of Act 1
print("shape of X_t @ w_t:", tuple(s_t.shape))

## <font color="#1967d2"><b>Step 11: The layer of Act 1, in PyTorch</b></font>

The line of Act 1 that applies a layer to all the points needs no change: PyTorch has the operator `@`, the transpose `.T` and broadcasting, with the meaning they have in NumPy.

In [ ]:
W_t = torch.from_numpy(W).float()        # the weights of the layer, as a float32 tensor
b_t = torch.from_numpy(b).float()        # the biases of the layer

S_t = X_t @ W_t.T + b_t                  # the line of Act 1, unchanged
print("shape of S_t:", tuple(S_t.shape))
print(f"largest difference from the NumPy scores S: {np.abs(S_t.numpy() - S).max():.1e}")

winner_t = S_t.argmax(dim=1)                     # NumPy: S.argmax(axis=1)
largest_t = S_t.max(dim=1, keepdim=True)         # NumPy: S.max(axis=1, keepdims=True)
print("shape of S_t.argmax(dim=1):", tuple(winner_t.shape))
print("S_t.max(dim=1, keepdim=True) returns two tensors: .values of shape", tuple(largest_t.values.shape),
      "and .indices of shape", tuple(largest_t.indices.shape))
print("torch.sigmoid of the first row of scores:", torch.sigmoid(S_t[0]))

**How to read the result.** The scores agree with those of NumPy to about seven digits, not exactly: the tensors hold `float32` numbers.

The two libraries, side by side:

| | NumPy | PyTorch |
|---|---|---|
| make an array from a list | `np.array([1.0, 2.0])` | `torch.tensor([1.0, 2.0])` |
| shape | `A.shape` | `A.shape` |
| the layer | `X @ W.T + b` | `X @ W.T + b` |
| a function of every entry | `np.exp(S)` | `torch.exp(S)` |
| along the rows | `axis=1`, `keepdims=True` | `dim=1`, `keepdim=True` |
| the largest entry of each row | `S.max(axis=1)` is an array | `S.max(dim=1)` holds `.values` and `.indices` |
| decimal numbers | `float64` | `float32` |
| where the numbers are | in the memory of the CPU | on a **device**: the CPU or a GPU (Step 14) |

**Your turn.** Ask an AI assistant to translate your function `softmax_rows` of Act 1 into PyTorch. Give the assistant your NumPy function, and state all of these requirements:

- the new function is named `softmax_rows_torch` and takes one argument, `S`, a PyTorch tensor with n rows and K columns;
- it returns a tensor of the same shape, in which every row is the softmax of the same row of `S`;
- it uses operations on whole tensors: **no loop**, and not PyTorch's own softmax function;
- like your function of Act 1, it subtracts from every row its largest entry before it takes the exponential.

Paste your prompt into the next cell, and the function into the code cell after it.

**Our prompt.** Double-click this cell and replace this line with the prompt you gave to the AI assistant.

In [ ]:
# Your turn: paste the function softmax_rows_torch here, then run this cell.

**Check.** The next cell tests your function on the three rows of scores of Act 1, and compares it with PyTorch's own `torch.softmax`.

In [ ]:
assert "softmax_rows_torch" in globals(), "Paste the function softmax_rows_torch into the cell above and run that cell."
assert not has_loop(softmax_rows_torch), "This function must use operations on whole tensors: no loop."
assert not mentions(softmax_rows_torch, ("softmax(", "Softmax", "scipy", "sklearn")), (
    "This function must compute the softmax itself: not torch.softmax, and no softmax function from another library.")

test = torch.tensor([[-1.0, 0.0, 2.0, 1.0],
                     [0.0, 0.0, 0.0, 0.0],
                     [1000.0, 1003.0, 1001.0, 1002.0]])
try:
    result = softmax_rows_torch(test.clone())
except TypeError:
    raise AssertionError(
        "A tensor was combined with something that is not a tensor. In PyTorch, S.max(dim=1, keepdim=True) returns "
        "two things, the largest values and their positions: take its .values.") from None
except RuntimeError:
    raise AssertionError(
        "The shapes inside softmax_rows_torch do not fit. The largest entry and the sum of every row must keep "
        "the shape (n, 1): use keepdim=True.") from None
assert isinstance(result, torch.Tensor), "softmax_rows_torch must return a PyTorch tensor: use torch.exp, not np.exp."
assert result.shape == test.shape, f"softmax_rows_torch returned a tensor of shape {tuple(result.shape)}; it must be {tuple(test.shape)}."
assert not torch.isnan(result).any(), (
    "softmax_rows_torch returns nan (not a number) for the row of very large scores. "
    "Subtract the largest entry of each row before the exponential.")
assert torch.allclose(result.sum(dim=1), torch.ones(3)), (
    "The rows of the result do not add to 1. The largest entry and the sum must be taken along each row: dim=1.")
assert torch.allclose(result, torch.softmax(test, dim=1), atol=1e-5), (
    f"The result is not the softmax of the rows. For the scores -1, 0, 2, 1 the function gives {result[0].tolist()}; "
    "meeting 8 found 0.03, 0.09, 0.64, 0.24.")
print("Correct: softmax_rows_torch agrees with PyTorch's own torch.softmax(S, dim=1), also for very large scores.")

**Your turn.** Put your function `softmax_rows` of Act 1 beside `softmax_rows_torch`, and compare them line by line. List in the next cell every difference that you find. **Review Quiz 11** asks for this list.

**Our answer.** Double-click this cell and replace this line with the differences between the NumPy function and the PyTorch function.

The next cell repaints your image as Step 7 of Act 1 did, with tensors in place of arrays. To see your own picture again, replace the paints with the paints of your group. `.numpy()` turns a tensor back into a NumPy array, which the drawing functions need.

In [ ]:
PAINT = np.array([[0.10, 0.07, 0.30],      # the paint of unit 0: replace these rows with the paints of your group
                  [0.80, 0.10, 0.35],
                  [0.98, 0.55, 0.15],
                  [0.99, 0.85, 0.40],
                  [1.00, 1.00, 0.92]])
T = 0.02                                   # the temperature of the softmax

PAINT_t = torch.from_numpy(PAINT).float()
Prob_t = softmax_rows_torch((X_t @ W_t.T + b_t) / T)     # scores, then probabilities
new_t = Prob_t @ PAINT_t                                  # probabilities, then colours
print("shape of new_t:", tuple(new_t.shape), "  type of its numbers:", new_t.dtype)
show([image, new_t.reshape(height, width, 3).numpy()], ["the image", "repainted with PyTorch"])

## <font color="#1967d2"><b>Step 12: PyTorch's own layer</b></font>

A layer of units is so common that PyTorch has it ready. `torch.nn.Linear(3, K)` makes a layer of K units for points with 3 coordinates, and fills it with random weights and biases. The next cell makes one, puts the weights and the biases of Act 1 into it, and applies it to all the points.

In [ ]:
linear = torch.nn.Linear(3, K)           # a layer of K units for points with 3 coordinates
print("shape of linear.weight:", tuple(linear.weight.shape), "  shape of linear.bias:", tuple(linear.bias.shape))

with torch.no_grad():                    # this cell needs no gradients
    linear.weight.copy_(W_t)             # the weights of Act 1 go into the layer
    linear.bias.copy_(b_t)               # and so do the biases
    S_linear = linear(X_t)               # the layer, applied to all the points

print("shape of linear(X_t):", tuple(S_linear.shape))
print(f"largest difference between linear(X_t) and X_t @ W_t.T + b_t: {float((S_linear - S_t).abs().max()):.1e}")

**How to read the result.** The weights of the layer have shape (K, 3): PyTorch stores **one unit in each row**, as the matrix `W` of Act 1 does. Its documentation gives the formula of this layer as $y = xA^{\mathsf T} + b$, where $A$ is the matrix of weights: the line of Act 1.

PyTorch can also compute the gradients that the descent of Unit I needs, for every weight of a network. `torch.no_grad()` says that this cell needs none. From Unit III on, networks are built from such layers and trained in this way.

## <font color="#1967d2"><b>Step 13: A second processor: the GPU</b></font>

The processor that has run every program of this course so far is the **CPU**, the central processing unit. A CPU has a few **cores**. Each core can run any program, one instruction after another, very fast.

A **GPU**, a graphics processing unit, is a second processor in the same computer. It was built for one kind of work: to compute the colours of the pixels of a screen, many times a second. That work is one short computation, repeated for millions of pixels, and the pixels do not depend on each other, as in Act 1. A GPU therefore has thousands of small cores that carry out the same instructions on different numbers at the same time, and it has a memory of its own.

A product of matrices is work of the same kind: each entry of the result is a sum of products that depends on no other entry. For that reason neural networks are computed on GPUs.

The page **[CPU and GPU](https://ikoutis.github.io/cs301-labs/unit-2/pages/cpu-and-gpu.html)** shows the two processors side by side. The next cell prints what this session has.

In [ ]:
print("logical cores of the CPU that this session can use:", os.cpu_count())
if GPU:
    gpu = torch.cuda.get_device_properties(0)
    print("the GPU:", gpu.name)
    print(f"memory of the GPU: {gpu.total_memory / 2 ** 30:.1f} GB")
else:
    print("This session has no GPU. Read the answers in the notebook of a member of your group whose session has one.")

## <font color="#1967d2"><b>Step 14: A tensor lives on a device</b></font>

Every tensor is stored on a **device**: in the memory of the CPU, or in the memory of the GPU. PyTorch names the GPU `"cuda"`, after CUDA, the system with which GPUs made by the company NVIDIA are programmed.

- `X_t.to("cuda")` **copies** the numbers of a tensor into the memory of the GPU and returns the copy.
- An operation needs all of its tensors on one device, and it is carried out by the processor of that device.
- `.cpu()` copies a result back into the memory of the CPU.

The line that computes the scores does not change.

In [ ]:
print("X_t is on the device:", X_t.device)
if GPU:
    X_g = X_t.to("cuda")                 # copies of the three tensors, in the memory of the GPU
    W_g = W_t.to("cuda")
    b_g = b_t.to("cuda")
    print("X_g is on the device:", X_g.device)

    try:
        X_g @ W_t.T                      # X_g is on the GPU and W_t is on the CPU
    except RuntimeError as error:
        print("X_g @ W_t.T fails:", error)

    S_g = X_g @ W_g.T + b_g              # the line of Act 1, carried out by the GPU
    print("S_g is on the device:", S_g.device)
    print(f"largest difference between the scores of the GPU and of the CPU: {float((S_g.cpu() - S_t).abs().max()):.1e}")
else:
    print("This session has no GPU, so every tensor stays in the memory of the CPU.")

## <font color="#1967d2"><b>Step 15: Measure the product on the CPU and on the GPU</b></font>

The layer on your image is a small product: its matrix `W` has 5 rows and 3 columns. The layers of real networks are far larger. The function `time_product` in the next cell measures `X @ W.T + b` for n points with m coordinates and a layer of K units, where n, m and K are all equal to `size`, and all the numbers are random.

Two details of the measurement are particular to a GPU:

- PyTorch hands a computation to the GPU and goes on at once, before the result is ready. `torch.cuda.synchronize()` waits until the GPU has finished. Without it, the clock would measure the handing over and not the work.
- The first use of a device takes extra time, so `warm_up` runs one small product that is not measured.

In [ ]:
def finish(device):
    """Wait until the device has finished its work. A CPU has always finished; a GPU may still be working."""
    if device == "cuda":
        torch.cuda.synchronize()


def time_product(size, device):
    """The average time, in seconds, of X @ W.T + b on the device, with n = m = K = size."""
    X = torch.rand(size, size, device=device)        # random float32 numbers, made on the device
    W = torch.rand(size, size, device=device)
    b = torch.rand(size, device=device)
    repeats = max(1, min(200, 8_000_000_000 // size ** 3))     # many repeats of a small product, one of a large one
    finish(device)
    start = time.perf_counter()
    for _ in range(repeats):
        S = X @ W.T + b
    finish(device)
    return (time.perf_counter() - start) / repeats


def warm_up(device):
    """One small product that is not measured."""
    time_product(64, device)


print("time_product and warm_up are defined.")

The next cell measures seven sizes, from 64 to 4,096, on each device of this session. At size 4,096 the product takes 4,096 × 4,096 × 4,096 multiplications, about 69 billion. The cell takes up to half a minute, most of it for the CPU at the largest size.

In [ ]:
sizes = [64, 128, 256, 512, 1024, 2048, 4096]
devices = ["cpu", "cuda"] if GPU else ["cpu"]
seconds = {}
for device in devices:
    warm_up(device)
    seconds[device] = [time_product(size, device) for size in sizes]

gpu_name = torch.cuda.get_device_name(0) if GPU else "none"
print(f"X @ W.T + b with n = m = K = size, in seconds. CPU: {os.cpu_count()} logical cores. GPU: {gpu_name}.")
print(f"{'size':>6s}{'multiplications':>18s}{'CPU':>12s}" + (f"{'GPU':>12s}{'CPU / GPU':>12s}" if GPU else ""))
for i, size in enumerate(sizes):
    line = f"{size:6d}{size ** 3:18,d}{seconds['cpu'][i]:12.6f}"
    if GPU:
        line += f"{seconds['cuda'][i]:12.6f}{seconds['cpu'][i] / seconds['cuda'][i]:12.1f}"
    print(line)

if GPU:
    X_cpu = torch.rand(4096, 4096)
    finish("cuda")
    start = time.perf_counter()
    X_gpu = X_cpu.to("cuda")
    finish("cuda")
    t_move = time.perf_counter() - start
    megabytes = X_cpu.element_size() * X_cpu.nelement() / 1e6
    print(f"\nCopying one matrix of size 4096 ({megabytes:.0f} MB) to the GPU takes {t_move:.4f} seconds; "
          f"the product at that size takes the GPU {seconds['cuda'][-1]:.4f} seconds.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(sizes, seconds["cpu"], marker="o", color="#2a78d6", label="CPU")
ax.annotate("CPU", (sizes[-1], seconds["cpu"][-1]), xytext=(8, 0), textcoords="offset points", va="center")
if GPU:
    ax.plot(sizes, seconds["cuda"], marker="s", linestyle="--", color="#eb6834", label="GPU")
    ax.annotate("GPU", (sizes[-1], seconds["cuda"][-1]), xytext=(8, 0), textcoords="offset points", va="center")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(sizes)
ax.set_xticklabels([str(size) for size in sizes])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, position: f"{value:g}"))
ax.margins(x=0.08)
ax.set_xlabel("size: the number of points, of coordinates and of units")
ax.set_ylabel("seconds for one product")
ax.set_title("The time of X @ W.T + b. Each step up the axis is 10 times longer.")
ax.grid(True, which="major", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

last, first = len(sizes) - 1, 0
print("Line for the discussion board:")
if GPU:
    print(f"GPU {gpu_name}, CPU with {os.cpu_count()} logical cores. Size 4096: CPU {seconds['cpu'][last]:.3f} s, "
          f"GPU {seconds['cuda'][last]:.4f} s, so the GPU is {seconds['cpu'][last] / seconds['cuda'][last]:.0f} times as fast. "
          f"Size 64: CPU {seconds['cpu'][first] * 1e6:.0f} microseconds, GPU {seconds['cuda'][first] * 1e6:.0f} microseconds.")
else:
    print(f"No GPU in this session. CPU with {os.cpu_count()} logical cores. Size 4096: {seconds['cpu'][last]:.3f} s. "
          f"Size 64: {seconds['cpu'][first] * 1e6:.0f} microseconds.")

**How to read the chart.** Both axes are stretched: each size is twice the size before it, and each mark on the vertical axis is ten times the mark below it. The table above the chart holds the same numbers.

**Your turn.** Answer in the next cell, from the table.

1. At which sizes is the CPU the faster one, and at which sizes the GPU?
2. From size 2,048 to size 4,096, by what factor does the time of the CPU grow? Act 3 explains this factor.

**Then post now.** Your group posts on the discussion board of this meeting, as a reply to its post of Act 1. The post contains:

- the line that the cell printed under the chart;
- the chart. To save it, right-click it and choose **Save image as**;
- under the chart, **one sentence that says what it shows**, so that a classmate who cannot see the chart still gets the result;
- one sentence on what surprised your group.

**Review Quiz 11** asks why a small product is not faster on the GPU, and what copying the data costs.

**Our answers.** Double-click this cell and replace this line with your two answers.

## Act 2 ends here

**Keep this notebook open.** Act 3 asks for one number of the table of Step 15: the seconds of the GPU at size 4,096. At the end of the meeting you upload this notebook to Canvas, with the notebooks of the other acts.

Act 3 has its own notebook. It is linked in the module of this meeting on Canvas, and here: [open the notebook of Act 3 in Colab](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-11-act-3.ipynb).